In [1]:
print("Hello, from my RTX 4090 workstation!")

Hello, from my RTX 4090 workstation!


In [2]:
import pandas as pd

# Load the existing driver-based splits
train = pd.read_csv("../data/splits/driver_based/train.csv")
val = pd.read_csv("../data/splits/driver_based/val.csv")
test = pd.read_csv("../data/splits/driver_based/test.csv")

# Collect driver IDs from each split
train_drivers = set(train["subject"])
val_drivers = set(val["subject"])
test_drivers = set(test["subject"])

# Check for overlapping drivers
print("Train drivers:", len(train_drivers))
print("Validation drivers:", len(val_drivers))
print("Test drivers:", len(test_drivers))

print("Train-Val overlap:", train_drivers & val_drivers)
print("Train-Test overlap:", train_drivers & test_drivers)
print("Val-Test overlap:", val_drivers & test_drivers)

Train drivers: 18
Validation drivers: 4
Test drivers: 4
Train-Val overlap: set()
Train-Test overlap: set()
Val-Test overlap: set()


In [3]:
from pathlib import Path
import pandas as pd

# Location of the labeled images
image_dir = Path("../data/raw/imgs/train")

# Combine our three dataset splits
df_all = pd.concat([train, val, test], ignore_index=True)

# Check whether each image exists
df_all["exists"] = df_all.apply(
    lambda row: (image_dir / row["classname"] / row["img"]).exists(),
    axis=1
)

print("Total images:", len(df_all))
print("Images found:", df_all["exists"].sum())
print("Missing images:", (~df_all["exists"]).sum())
print("Duplicate images:", df_all["img"].duplicated().sum())

Total images: 22424
Images found: 22424
Missing images: 0
Duplicate images: 0


In [4]:
# Load the existing random splits
random_train = pd.read_csv("../data/splits/random/train.csv")
random_val = pd.read_csv("../data/splits/random/val.csv")
random_test = pd.read_csv("../data/splits/random/test.csv")

# Count images and unique drivers in each split
for name, df in [
    ("Train", random_train),
    ("Validation", random_val),
    ("Test", random_test)
]:
    print(f"{name}: {len(df)} images, {df['subject'].nunique()} drivers")

# Check driver overlap between training and validation
overlap = set(random_train["subject"]) & set(random_val["subject"])

print("\nDrivers shared between train and validation:", len(overlap))

# Verify that random splits contain no duplicate images
random_all = pd.concat(
    [random_train, random_val, random_test],
    ignore_index=True
)

print("Total images:", len(random_all))
print("Duplicate images:", random_all["img"].duplicated().sum())

# Confirm both splitting strategies use the same images
print("Same images in both protocols:",
      set(random_all["img"]) == set(df_all["img"]))

Train: 15696 images, 26 drivers
Validation: 3364 images, 26 drivers
Test: 3364 images, 26 drivers

Drivers shared between train and validation: 26
Total images: 22424
Duplicate images: 0
Same images in both protocols: True
